# 04 · WEEKLY MOMENTUM · RECONCILIATION
**Alphaca Capital** · `WEEKLY_MOM.ethusd.cfd.v1` · variant `v2_ethusd_long` · vehicle: Axi ETHUSD CFD, H1 candles, MT5 hedging account.

Reconciles the MT5 Expert Advisor (`Weekly_Momentum_ETH-USD_Axi.mq5`) against `weekly_momentum_engine.py`, one layer at a time:

| Layer | EA file | What is compared with the engine |
| :--- | :--- | :--- |
| 1. Signals | `weekly_momentum_log.csv` | One row per Monday slot: momentum, BUY / SKIP decision and reason |
| 2. Trades | `weekly_momentum_trades.csv` | One row per closed trade: entry / exit time, entry / exit price, net PnL, swap, lots |

**How to produce the EA files**: Strategy Tester on a hedging account, symbol ETHUSD, H1, every tick based on real ticks, `InpCapitalBase = WM_CAP_FIXED` with `InpFixedCapital = CAPITAL`, all rule inputs untouched. Copy both CSVs from `Common\Files` into `data/Reconcilation/Weekly_Momentum/`. The engine must read the same Axi H1 history the tester used.

In [1]:
from pathlib import Path

VARIANT = "v2_ethusd_long"
HARD_LIMIT = 0.09
CAPITAL = 3_500.0

EA_CAPITAL_MODE = "fixed"
RECON_DIR = Path("../../data/Reconcilation/Weekly_Momentum")
LOG_FILE = RECON_DIR / "weekly_momentum_log.csv"
DEALS_FILE = RECON_DIR / "weekly_momentum_trades.csv"

## CONFIG

In [2]:
import sys
import importlib
from pathlib import Path
import numpy as np
import pandas as pd

def find_upwards(name, start=None, must_be_dir=True):
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        target = base / name
        if target.is_dir() if must_be_dir else target.exists():
            return base
    raise FileNotFoundError(f"Could not find '{name}' walking up from {here}")

EDGE_DIR = find_upwards("variants")
utils_path = str(find_upwards("utils") / "utils")
for p in (utils_path, str(EDGE_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import weekly_momentum_engine as eng
import variant_config
from variant_config import load_variant

importlib.reload(eng)

variant_config.EDGE_DIR = EDGE_DIR
assert (variant_config.EDGE_DIR / "variants").is_dir(), f"Doesn't exist {variant_config.EDGE_DIR / 'variants'}"

cfg = load_variant(VARIANT)
eng.set_engine_defaults(**cfg.get("engine_defaults", {}))
TICKER = cfg["instrument"]["ticker"]
DATA_START = cfg["instrument"]["start"]
P = cfg["params"]
LOOKBACK, HOLD, ENTRY_DOW, ENTRY_HOUR = P["lookback_days"], P["hold_days"], P["entry_dow"], P["entry_hour"]
DIRECTION, COST_MODE, COST_BPS = P["direction"], P["cost_mode"], P["cost_bps_per_side"]
SIZE_FRAC = eng.ENGINE_DEFAULTS["size_fraction"]
CONTRACT_SIZE, MIN_LOT, LOT_STEP = (eng.ENGINE_DEFAULTS[k] for k in ("contract_size", "min_lot", "lot_step"))

RULES = dict(lookback_days=LOOKBACK, hold_days=HOLD, entry_dow=ENTRY_DOW, entry_hour=ENTRY_HOUR,
             direction=DIRECTION, cost_mode=COST_MODE, cost_bps_per_side=COST_BPS)

EA_INPUTS = dict(lookback_days=31, hold_days=11, direction="long_only", entry_dow_mt5=1, entry_hour=0, size_fraction=0.05)

TOL_MOM_PP = 1e-3
TOL_ENTRY_BPS = 8.0
TOL_EXIT_BPS = 3.0
TOL_TIME_SEC = 120
TOL_NET_PP = 0.25

CHECKS = []

def check(section, name, passed, detail="", quiet=False):
    status = "SKIP" if passed is None else ("PASS" if bool(passed) else "FAIL")
    entry = {"section": section, "name": name, "status": status, "detail": detail}
    for i, c in enumerate(CHECKS):
        if c["section"] == section and c["name"] == name:
            CHECKS[i] = entry
            break
    else:
        CHECKS.append(entry)
    if not quiet:
        print(f"   [{status}] {name}" + (f"  ->  {detail}" if detail else ""))
    return status

print("Imports complete.")
print(f"Loaded weekly_momentum_engine from: {eng.__file__}")
print(f"Variant: {cfg['id']} | {TICKER} | momentum {LOOKBACK} d -> hold {HOLD} d | {DIRECTION} | size {SIZE_FRAC:.0%} | capital {CAPITAL:,.0f}")
print(f"EA capital mode: {EA_CAPITAL_MODE}")
print("CHECKS (config):")
engine_dow_mt5 = (ENTRY_DOW + 1) % 7
check("R0. Config", "EA inputs mirror the variant",
      (EA_INPUTS["lookback_days"], EA_INPUTS["hold_days"], EA_INPUTS["direction"], EA_INPUTS["entry_dow_mt5"],
       EA_INPUTS["entry_hour"], EA_INPUTS["size_fraction"])
      == (LOOKBACK, HOLD, DIRECTION, engine_dow_mt5, ENTRY_HOUR, SIZE_FRAC),
      f"EA {EA_INPUTS} | engine lookback {LOOKBACK}, hold {HOLD}, {DIRECTION}, MT5 weekday {engine_dow_mt5}, hour {ENTRY_HOUR}, size {SIZE_FRAC}")

Imports complete.
Loaded weekly_momentum_engine from: /Users/luyandy/Documents/Alphaca Capital/Alphaca-Capital-Research/research_development/02_strategy_development/ETHUSD_Weekly_Momentum/weekly_momentum_engine.py
Variant: WEEKLY_MOM.ethusd.cfd.v1 | ethusd | momentum 31 d -> hold 11 d | long_only | size 5% | capital 3,500
EA capital mode: fixed
CHECKS (config):
   [PASS] EA inputs mirror the variant  ->  EA {'lookback_days': 31, 'hold_days': 11, 'direction': 'long_only', 'entry_dow_mt5': 1, 'entry_hour': 0, 'size_fraction': 0.05} | engine lookback 31, hold 11, long_only, MT5 weekday 1, hour 0, size 0.05


'PASS'

## 1. ENGINE REFERENCE
The engine runs on the full history with the frozen rules. Its Monday slots are re-keyed from UTC to the **MT5 server date** (`srv_time`), which is the key the EA writes in its log.

In [3]:
df_raw = eng.load_data(TICKER, start=DATA_START)
df = eng.prepare(df_raw, lookback_days=LOOKBACK, entry_dow=ENTRY_DOW, entry_hour=ENTRY_HOUR)
trades_all = eng.run_weekly_backtest(df, **RULES)
sizes = eng.cfd_trade_sizes(trades_all, CAPITAL, SIZE_FRAC)

srv = pd.to_datetime(df["srv_time"])
last_srv = srv.iloc[-1]

ref = df.loc[df["entry_ok"], ["mom_ret", "open", "spread_px"]].copy()
ref["entry_utc"] = ref.index
ref["slot"] = srv.loc[ref.index].dt.normalize().to_numpy()
tr_by_utc = trades_all.assign(entry_time=pd.to_datetime(trades_all["entry_time"])).set_index("entry_time")
ref["take"] = ref.index.isin(tr_by_utc.index)
ref["side"] = tr_by_utc["side"].reindex(ref.index).to_numpy()
ref = ref.set_index("slot")
assert ref.index.is_unique, "Two engine entry candles map to the same server date."

print(len(df_raw), "hourly bars |", df_raw.index[0], "->", df_raw.index[-1])
print(f"Engine entry slots: {len(ref)} | trades: {len(trades_all)} | last server timestamp: {last_srv}")
print(f"Weeks not traded: {trades_all.attrs['skipped']}")

68722 hourly bars | 2019-01-01 00:00:00 -> 2026-11-06 10:00:00
Engine entry slots: 408 | trades: 221 | last server timestamp: 2026-11-06 12:00:00
Weeks not traded: {'candidates': 408, 'warmup': 4, 'no_signal_bar': 0, 'zero_signal': 0, 'no_exit': 1, 'direction_filter': 182, 'trend_warmup': 0, 'trend_filter': 0}


## 2. SIGNAL RECONCILIATION (`weekly_momentum_log.csv`)
Compared on every Monday slot inside the window the EA actually ran, **including the skipped weeks** (the EA logs the momentum of the weeks it skips as well).

Slots whose exit would fall after the last engine candle are labelled `open_at_end`: the EA buys them, the engine cannot close them, so they are expected to differ and are excluded from the decision check.

In [4]:
log = pd.read_csv(LOG_FILE, sep=";", header=None,
                  names=["slot", "spread_pts", "decision", "reason", "mom_pct", "lots", "price"],
                  dtype={"slot": str, "price": str})
log["slot"] = pd.to_datetime(log["slot"])
n_dup = int(log["slot"].duplicated().sum())
log = log.drop_duplicates("slot", keep="last").set_index("slot").sort_index()

decimals = log["price"].dropna().str.split(".").str[1].str.len()
DIGITS = int(decimals.max()) if len(decimals) else 2
POINT = 10.0 ** -DIGITS
log["price"] = pd.to_numeric(log["price"], errors="coerce")
log["spread_px_ea"] = log["spread_pts"] * POINT

w0, w1 = log.index.min(), log.index.max()
ref_w = ref[(ref.index >= w0) & (ref.index <= w1)]
j = ref_w.merge(log, left_index=True, right_index=True, how="outer", indicator=True)
j["open_at_end"] = (j.index + pd.Timedelta(days=HOLD)) > last_srv

eng_taken = j["take"].fillna(False).astype(bool)
j["eng_dec"] = np.where(eng_taken, np.where(j["side"] > 0, "BUY", "SELL"), "SKIP")
j.loc[j["_merge"] == "right_only", "eng_dec"] = np.nan

both = j[j["_merge"] == "both"].copy()
mom_pair = both.dropna(subset=["mom_ret", "mom_pct"])
MOM_SCALE = min((1.0, 100.0), key=lambda s: (mom_pair["mom_ret"] * s - mom_pair["mom_pct"]).abs().max())
both["d_mom_pp"] = (both["mom_ret"] * MOM_SCALE - both["mom_pct"]).abs()

cmp = both[~both["open_at_end"]]
bad = cmp[cmp["eng_dec"] != cmp["decision"]]
only_eng = j[j["_merge"] == "left_only"]
only_ea = j[j["_merge"] == "right_only"]

print("=" * 70)
print("SIGNAL RECONCILIATION")
print("=" * 70)
print(f"EA window: {w0:%Y-%m-%d} -> {w1:%Y-%m-%d} | EA slots: {len(log)} | engine slots in window: {len(ref_w)}")
print(f"Symbol digits inferred from the log: {DIGITS} (point {POINT:g}) | engine momentum scale to match EA percent: x{MOM_SCALE:g}")
print(f"EA BUY: {(log['decision'] == 'BUY').sum()} | engine trades in window: {int(eng_taken.sum())} | open_at_end slots: {int(both['open_at_end'].sum())}")
print(j["_merge"].value_counts().to_string())
print()
print("Engine decision vs EA reason (excluding open_at_end):")
print(pd.crosstab(cmp["eng_dec"], cmp["reason"]).to_string())
print()
print("CHECKS (signals):")
check("R1. Signals", "EA log has one row per entry slot", n_dup == 0, f"{n_dup} duplicated slots (kept the last)")
check("R1. Signals", "Every engine entry slot is in the EA log", len(only_eng) == 0, f"{len(only_eng)} engine slots missing in the EA log")
check("R1. Signals", "Every EA slot is a valid engine entry candle", len(only_ea) == 0, f"{len(only_ea)} EA slots the engine has no entry candle for")
check("R1. Signals", f"Momentum matches (max diff <= {TOL_MOM_PP:g} pp)", both["d_mom_pp"].max() <= TOL_MOM_PP,
      f"max diff {both['d_mom_pp'].max():.6f} pp | slots above tolerance: {int((both['d_mom_pp'] > TOL_MOM_PP).sum())}")
check("R1. Signals", "Same decision on every slot", len(bad) == 0, f"{len(bad)} of {len(cmp)} slots differ")

print("\nSlots only on one side:")
display(j.loc[j["_merge"] != "both", ["take", "side", "decision", "reason", "_merge"]])
print("Different decisions:")
display(bad[["mom_ret", "mom_pct", "take", "side", "eng_dec", "decision", "reason"]])
print("Momentum above tolerance:")
display(both.loc[both["d_mom_pp"] > TOL_MOM_PP, ["mom_ret", "mom_pct", "d_mom_pp", "eng_dec", "decision"]])

FileNotFoundError: [Errno 2] No such file or directory: '../../data/Reconcilation/Weekly_Momentum/weekly_momentum_log.csv'

## 3. TRADE RECONCILIATION (`weekly_momentum_trades.csv`)
Engine trades are re-keyed by the server date of their entry and matched with the EA closed trades.

* **Time**: the EA fills at the first tick of the candle, so the lag against the server candle open should be seconds.
* **Price**: engine entry for a long is `open + spread` of the entry candle (CSV prices are BID), exit is the `open` of the exit candle. The EA fills at the real ask / bid of the first tick.
* **Net PnL**: EA `(profit + swap + commission) / (price_in x lots x contract)` in percent, against the engine `pnl`. The decomposition (price move after spread, swap) shows where any gap comes from.
* **Lots**: only exact when the EA ran with fixed capital; with equity sizing the lots compound and the check is `SKIP`.

In [ ]:
deals = pd.read_csv(DEALS_FILE, sep=";", parse_dates=["time_in", "time_out"])
deals["slot"] = deals["time_in"].dt.normalize()
n_dup_d = int(deals["slot"].duplicated().sum())
deals = deals.drop_duplicates("slot", keep="last")

i0 = trades_all["i_ent"].astype(int).to_numpy()
i1 = trades_all["i_sal"].astype(int).to_numpy()
et = trades_all.copy()
et["srv_in"] = srv.iloc[i0].to_numpy()
et["srv_out"] = srv.iloc[i1].to_numpy()
et["open_in"] = df["open"].iloc[i0].to_numpy()
et["spread_px_eng"] = df["spread_px"].iloc[i0].to_numpy()
et["slot"] = pd.to_datetime(et["srv_in"]).dt.normalize()
lots_tbl = sizes.loc[sizes["taken"], "lots"] if "taken" in sizes.columns else sizes["lots"]
assert len(lots_tbl) == len(et), f"cfd_trade_sizes returned {len(lots_tbl)} rows for {len(et)} trades."
et["lots_eng"] = lots_tbl.to_numpy()
et = et.set_index("slot")

ea_end = deals["time_out"].max()
in_window = (et.index >= w0) & (pd.to_datetime(et["srv_out"]) <= ea_end + pd.Timedelta(hours=1))
et_w = et[in_window]

m = et_w.merge(deals.set_index("slot"), left_index=True, right_index=True, how="outer", indicator=True)
mb = m[m["_merge"] == "both"].copy()

mb["lag_in_s"] = (mb["time_in"] - pd.to_datetime(mb["srv_in"])).dt.total_seconds()
mb["lag_out_s"] = (mb["time_out"] - pd.to_datetime(mb["srv_out"])).dt.total_seconds()
mb["exit_day_ok"] = mb["time_out"].dt.normalize() == pd.to_datetime(mb["srv_out"]).dt.normalize()
mb["d_entry_bps"] = (mb["price_in"] / mb["entry_price"] - 1.0) * 1e4
mb["d_exit_bps"] = (mb["price_out"] / mb["exit_price"] - 1.0) * 1e4
notional = mb["price_in"] * mb["lots"] * CONTRACT_SIZE
sign = np.where(mb["type"] == "long", 1.0, -1.0)
mb["ea_net_pct"] = (mb["profit"] + mb["swap"] + mb["commission"]) / notional * 100.0
mb["ea_swap_pct"] = mb["swap"] / notional * 100.0
mb["ea_px_pct"] = sign * (mb["price_out"] / mb["price_in"] - 1.0) * 100.0
mb["eng_px_pct"] = mb["gross_pct"] - mb["cost_pct"]
mb["d_net_pp"] = mb["ea_net_pct"] - mb["pnl"]
mb["d_px_pp"] = mb["ea_px_pct"] - mb["eng_px_pct"]
mb["d_swap_pp"] = mb["ea_swap_pct"] - mb["swap_pct"]
mb["fill_spread_ea"] = mb["price_in"] - mb["open_in"]

only_eng_t = m[m["_merge"] == "left_only"]
only_ea_t = m[m["_merge"] == "right_only"]

ea_usd = mb["profit"] + mb["swap"] + mb["commission"]
eng_usd_same_lots = mb["pnl"] / 100.0 * mb["price_in"] * mb["lots"] * CONTRACT_SIZE

events = pd.DataFrame({"t": list(deals["time_in"]) + list(deals["time_out"]),
                       "d": [1] * len(deals) + [-1] * len(deals)}).sort_values(["t", "d"])
conc_ea = int(events["d"].cumsum().max())
conc_eng = int(eng.max_concurrent(trades_all))
max_conc = -(-HOLD // 7)

print("=" * 70)
print("TRADE RECONCILIATION")
print("=" * 70)
print(f"EA closed trades: {len(deals)} | engine trades closed inside the EA window: {len(et_w)} | matched: {len(mb)}")
print(f"Entry lag vs candle open: median {mb['lag_in_s'].median():.0f} s, max {mb['lag_in_s'].max():.0f} s | exit lag: median {mb['lag_out_s'].median():.0f} s, max {mb['lag_out_s'].max():.0f} s")
print(f"Entry price diff (bps): mean {mb['d_entry_bps'].mean():+.2f}, max abs {mb['d_entry_bps'].abs().max():.2f} | exit price diff (bps): mean {mb['d_exit_bps'].mean():+.2f}, max abs {mb['d_exit_bps'].abs().max():.2f}")
print(f"Entry spread paid, price units: EA fill median {mb['fill_spread_ea'].median():.3f} vs engine candle median {mb['spread_px_eng'].median():.3f}")
print(f"Net pnl per trade (pp of notional): EA mean {mb['ea_net_pct'].mean():+.3f} | engine mean {mb['pnl'].mean():+.3f} | mean diff {mb['d_net_pp'].mean():+.3f}, max abs {mb['d_net_pp'].abs().max():.3f}")
print(f"   price move after spread: mean diff {mb['d_px_pp'].mean():+.3f} pp | swap: EA mean {mb['ea_swap_pct'].mean():+.3f} vs engine mean {mb['swap_pct'].mean():+.3f} pp | commission total (EA): {mb['commission'].sum():+.2f}")
print(f"Total net pnl, same lots: EA {ea_usd.sum():+,.2f} | engine {eng_usd_same_lots.sum():+,.2f} | diff {ea_usd.sum() - eng_usd_same_lots.sum():+,.2f}")
print(f"Max concurrent trades: EA {conc_ea} | engine {conc_eng} | expected at most {max_conc}")
print()
print("CHECKS (trades):")
check("R2. Trades", "EA log has one closed trade per slot", n_dup_d == 0, f"{n_dup_d} duplicated slots (kept the last)")
check("R2. Trades", "Every engine trade closed in the window has an EA trade", len(only_eng_t) == 0, f"{len(only_eng_t)} engine trades without an EA trade")
check("R2. Trades", "Every EA trade is an engine trade", len(only_ea_t) == 0, f"{len(only_ea_t)} EA trades the engine did not take")
check("R2. Trades", f"Entry fills within {TOL_TIME_SEC} s of the candle open", mb["lag_in_s"].abs().max() <= TOL_TIME_SEC, f"max lag {mb['lag_in_s'].abs().max():.0f} s")
check("R2. Trades", "Exit happens on the engine exit day", bool(mb["exit_day_ok"].all()), f"{int((~mb['exit_day_ok']).sum())} trades exit on another day")
check("R2. Trades", f"Exit fills within {TOL_TIME_SEC} s of the candle open", mb["lag_out_s"].abs().max() <= TOL_TIME_SEC, f"max lag {mb['lag_out_s'].abs().max():.0f} s")
check("R2. Trades", f"Entry price within {TOL_ENTRY_BPS:g} bps", mb["d_entry_bps"].abs().max() <= TOL_ENTRY_BPS, f"max {mb['d_entry_bps'].abs().max():.2f} bps")
check("R2. Trades", f"Exit price within {TOL_EXIT_BPS:g} bps", mb["d_exit_bps"].abs().max() <= TOL_EXIT_BPS, f"max {mb['d_exit_bps'].abs().max():.2f} bps")
check("R2. Trades", f"Net pnl per trade within {TOL_NET_PP:g} pp", mb["d_net_pp"].abs().max() <= TOL_NET_PP, f"max {mb['d_net_pp'].abs().max():.3f} pp | mean {mb['d_net_pp'].mean():+.3f} pp")
check("R2. Trades", "Engine pnl is not better than the EA on average (no hidden optimism)", mb["d_net_pp"].mean() >= -TOL_NET_PP / 2, f"mean EA minus engine {mb['d_net_pp'].mean():+.3f} pp")
if EA_CAPITAL_MODE == "fixed":
    lot_diff = (mb["lots"] - mb["lots_eng"]).abs()
    check("R2. Trades", "Lots equal the engine sizing", bool((lot_diff < 1e-9).all()), f"{int((lot_diff >= 1e-9).sum())} trades with different lots | max diff {lot_diff.max():g}")
else:
    check("R2. Trades", "Lots equal the engine sizing", None, "EA ran with equity sizing: lots compound, not comparable")
check("R2. Trades", f"Concurrent trades never above {max_conc}", conc_ea <= max_conc, f"EA max {conc_ea}")

print("\nTrades only on one side:")
display(m.loc[m["_merge"] != "both", ["side", "entry_price", "exit_price", "type", "price_in", "price_out", "lots", "_merge"]])
print("Largest net pnl gaps:")
display(mb.loc[mb["d_net_pp"].abs().sort_values(ascending=False).index[:10],
               ["pnl", "ea_net_pct", "d_net_pp", "d_px_pp", "d_swap_pp", "d_entry_bps", "d_exit_bps", "lots", "lots_eng"]])

## 4. VERDICT
`RECONCILED` only when every check is `PASS` or `SKIP`. Any `FAIL` is listed with its detail so the cause is investigated before the EA goes to the live account.

In [ ]:
status_df = pd.DataFrame(CHECKS)
print("=" * 70)
print("RECONCILIATION SUMMARY")
print("=" * 70)
for section, grp in status_df.groupby("section", sort=True):
    print(f"\n{section}")
    for _, r in grp.iterrows():
        print(f"   [{r['status']}] {r['name']}" + (f"  ->  {r['detail']}" if r["detail"] else ""))
n_fail = int((status_df["status"] == "FAIL").sum())
n_pass = int((status_df["status"] == "PASS").sum())
n_skip = int((status_df["status"] == "SKIP").sum())
print("\n" + "=" * 70)
print(f"PASS {n_pass} | FAIL {n_fail} | SKIP {n_skip}")
print("VERDICT: " + ("RECONCILED" if n_fail == 0 else "REVIEW - the failed checks above must be explained before going live"))
print("=" * 70)